# GOV-01 E8: pothole/manhole specialist — Kaggle version

Before running: attach the private Kaggle Dataset containing `v2_multilabel_n_rdd_expansion.zip`, then select **Settings → Accelerator → GPU**. This notebook uses no Google Drive and never loads the protected test split.


In [ ]:
# Locate the attached Kaggle data. It may be a ZIP or an already-extracted folder.
from pathlib import Path
import csv
import json
import math
import shutil
import zipfile

DATASET_FOLDER = 'v2_multilabel_n_rdd_expansion'
SPECIALIST_CONDITIONS = ('pothole', 'manhole_cover')
IMAGE_SIZE = (320, 320)
BATCH_SIZE = 16
SEED = 42
FOCAL_GAMMA = 2.0
POSITIVE_MULTIPLIER = 1.50
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
LOCAL_DATA_ROOT = WORK_ROOT / DATASET_FOLDER
OUTPUT_DIRECTORY = WORK_ROOT / 'v2_multilabel_e8_pothole_manhole_specialist_output'

def has_training_layout(root):
    return all((root / split / 'images').is_dir() and (root / split / 'labels.csv').is_file() for split in ('train', 'validation'))

attached_folders = [path for path in INPUT_ROOT.rglob(DATASET_FOLDER) if path.is_dir() and has_training_layout(path)]
attached_archives = list(INPUT_ROOT.rglob(f'{DATASET_FOLDER}.zip'))
if len(attached_folders) == 1:
    DATA_ROOT = attached_folders[0]
    ARCHIVE_PATH = None
    print('Using attached extracted Kaggle dataset:', DATA_ROOT)
elif len(attached_archives) == 1:
    ARCHIVE_PATH = attached_archives[0]
    DATA_ROOT = LOCAL_DATA_ROOT
    print('Using attached Kaggle ZIP:', ARCHIVE_PATH)
else:
    raise FileNotFoundError('Attach exactly one Kaggle Dataset containing v2_multilabel_n_rdd_expansion.zip or the extracted v2_multilabel_n_rdd_expansion folder.')

print('Protected test is deliberately not loaded by E8.')


In [ ]:
# Extract only training and validation data when a ZIP is attached; otherwise use the attached extracted dataset.
def split_is_complete(dataset_root, split_name):
    split_root = dataset_root / split_name
    labels_path = split_root / 'labels.csv'
    if not labels_path.is_file() or not (split_root / 'images').is_dir():
        return False
    with labels_path.open(newline='', encoding='utf-8') as source:
        rows = list(csv.DictReader(source))
    return bool(rows) and all((split_root / Path(row['materialized_image'].replace(chr(92), '/'))).is_file() for row in rows)

def training_data_is_complete(dataset_root):
    return all(split_is_complete(dataset_root, split_name) for split_name in ('train', 'validation'))

def assert_safe_members(members, destination_root):
    destination_root = destination_root.resolve()
    for member in members:
        destination = (destination_root / member.filename).resolve()
        if destination != destination_root and destination_root not in destination.parents:
            raise RuntimeError(f'Unsafe ZIP member: {member.filename}')

if ARCHIVE_PATH is not None:
    if training_data_is_complete(LOCAL_DATA_ROOT):
        print('Using verified temporary Kaggle extraction:', LOCAL_DATA_ROOT)
    else:
        if LOCAL_DATA_ROOT.exists():
            shutil.rmtree(LOCAL_DATA_ROOT)
        print('Extracting training and validation data. The protected test folder is not extracted.')
        train_prefix, validation_prefix = f'{DATASET_FOLDER}/train/', f'{DATASET_FOLDER}/validation/'
        with zipfile.ZipFile(ARCHIVE_PATH) as archive:
            members = [member for member in archive.infolist() if member.filename.startswith(train_prefix) or member.filename.startswith(validation_prefix)]
            if not members:
                raise RuntimeError('The attached ZIP has no training/validation folders at the expected location.')
            assert_safe_members(members, WORK_ROOT)
            archive.extractall(WORK_ROOT, members=members)
    if not training_data_is_complete(LOCAL_DATA_ROOT):
        raise RuntimeError('Temporary training data is incomplete. Restart the notebook and rerun from the first cell.')

if not training_data_is_complete(DATA_ROOT):
    raise RuntimeError(f'Training/validation data is incomplete under {DATA_ROOT}')
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
print('Training data root:', DATA_ROOT)
print('Kaggle output folder:', OUTPUT_DIRECTORY)


In [ ]:
# Read the two specialist labels. Unknown labels are not treated as negative labels.
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score, precision_recall_fscore_support

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPU devices:', gpus)
if not gpus:
    raise RuntimeError('No GPU is connected. In Kaggle open Settings and select Accelerator → GPU, then restart and run from the first cell.')

def read_split(split_name):
    split_root = DATA_ROOT / split_name
    paths, labels, masks, sources = [], [], [], []
    with (split_root / 'labels.csv').open(newline='', encoding='utf-8') as source:
        for row in csv.DictReader(source):
            image_path = split_root / Path(row['materialized_image'].replace(chr(92), '/'))
            if not image_path.is_file():
                raise FileNotFoundError(f'Missing materialized image: {image_path}')
            label_row, mask_row = [], []
            for condition in SPECIALIST_CONDITIONS:
                known = int(row[f'{condition}_known'])
                value = row[f'{condition}_present']
                if known not in (0, 1) or (known == 0 and value != '') or (known == 1 and value not in ('0', '1')):
                    raise ValueError(f'Invalid masked label: {row["record_id"]} / {condition}')
                label_row.append(float(value) if known else 0.0)
                mask_row.append(float(known))
            paths.append(str(image_path)); labels.append(label_row); masks.append(mask_row); sources.append(row['source_dataset'])
    return np.asarray(paths), np.asarray(labels, dtype='float32'), np.asarray(masks, dtype='float32'), np.asarray(sources)

train_paths, train_labels, train_masks, train_sources = read_split('train')
validation_paths, validation_labels, validation_masks, validation_sources = read_split('validation')
train_eligible = train_masks.sum(axis=1) > 0
validation_eligible = validation_masks.sum(axis=1) > 0
train_paths, train_labels, train_masks, train_sources = train_paths[train_eligible], train_labels[train_eligible], train_masks[train_eligible], train_sources[train_eligible]
validation_paths, validation_labels, validation_masks = validation_paths[validation_eligible], validation_labels[validation_eligible], validation_masks[validation_eligible]
for name, labels, masks in [('Train', train_labels, train_masks), ('Validation', validation_labels, validation_masks)]:
    print(f'\n{name}: {len(labels)} specialist-eligible images')
    for index, condition in enumerate(SPECIALIST_CONDITIONS):
        print(f'{condition:16} known={int(masks[:, index].sum()):5} positive={int((labels[:, index] * masks[:, index]).sum()):5}')

def load_image(path):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    return tf.cast(image, tf.float32)


In [ ]:
# Use moderate source balancing plus additional pothole/manhole-positive examples in each epoch.
source_names, source_counts = np.unique(train_sources, return_counts=True)
source_count_map = dict(zip(source_names, source_counts))
source_weight = np.asarray([1.0 / math.sqrt(source_count_map[source]) for source in train_sources], dtype='float64')
target_positive = np.any((train_labels == 1.0) & (train_masks == 1.0), axis=1)
sampling_weight = source_weight * np.where(target_positive, 1.75, 1.0)
sampling_weight = sampling_weight / sampling_weight.sum()
sampled_indices = np.random.default_rng(SEED).choice(len(train_paths), size=len(train_paths), replace=True, p=sampling_weight)
sampling_summary = {
    'strategy': 'inverse-square-root source balancing plus 1.75x positive pothole/manhole sampling weight',
    'eligible_training_images': int(len(train_paths)), 'sampled_images_per_epoch': int(len(sampled_indices)),
    'positive_pothole_or_manhole_in_original_pool': int(target_positive.sum()),
    'positive_pothole_or_manhole_in_sampled_pool': int(target_positive[sampled_indices].sum()),
    'original_source_counts': {str(name): int(count) for name, count in zip(source_names, source_counts)}
}
(OUTPUT_DIRECTORY / 'e8_sampling_summary.json').write_text(json.dumps(sampling_summary, indent=2) + '\n')
print(json.dumps(sampling_summary, indent=2))

def make_dataset(paths, labels, masks, training):
    packed = np.concatenate([labels, masks], axis=1)
    dataset = tf.data.Dataset.from_tensor_slices((paths, packed))
    if training:
        dataset = dataset.shuffle(min(len(paths), 8192), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(lambda path, target: (load_image(path), target), num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset(train_paths[sampled_indices], train_labels[sampled_indices], train_masks[sampled_indices], training=True)
validation_dataset = make_dataset(validation_paths, validation_labels, validation_masks, training=False)
validation_images = tf.data.Dataset.from_tensor_slices(validation_paths).map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
# Stage 1: train a new two-label head while EfficientNetV2B0 stays frozen.
@tf.keras.utils.register_keras_serializable(package='gov01')
def masked_specialist_focal_loss(packed_targets, probabilities):
    count = len(SPECIALIST_CONDITIONS)
    labels, masks = packed_targets[:, :count], packed_targets[:, count:]
    probabilities = tf.clip_by_value(probabilities, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
    base_loss = tf.keras.backend.binary_crossentropy(labels, probabilities)
    probability_of_truth = labels * probabilities + (1.0 - labels) * (1.0 - probabilities)
    focal_factor = tf.pow(1.0 - probability_of_truth, FOCAL_GAMMA)
    positive_weight = 1.0 + (POSITIVE_MULTIPLIER - 1.0) * labels
    weighted_loss = base_loss * focal_factor * positive_weight * masks
    return tf.math.divide_no_nan(tf.reduce_sum(weighted_loss, axis=1), tf.reduce_sum(masks, axis=1))

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', seed=SEED),
    tf.keras.layers.RandomRotation(0.03, seed=SEED),
    tf.keras.layers.RandomContrast(0.08, seed=SEED),
], name='specialist_training_augmentation')
base = tf.keras.applications.EfficientNetV2B0(include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3), include_preprocessing=True)
base.trainable = False
inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='road_image')
x = augmentation(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.35, seed=SEED)(x)
outputs = tf.keras.layers.Dense(len(SPECIALIST_CONDITIONS), activation='sigmoid', name='specialist_probabilities')(x)
model = tf.keras.Model(inputs, outputs, name='v2_multilabel_e8_pothole_manhole_specialist')
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=8e-4), loss=masked_specialist_focal_loss)
head_history = model.fit(train_dataset, validation_data=validation_dataset, epochs=5, verbose=1)


In [ ]:
# Stage 2: fine-tune final layers and save the model with the strongest pothole/manhole validation F1.
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
for layer in base.layers[-40:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=8e-6), loss=masked_specialist_focal_loss)

class SpecialistValidationF1(tf.keras.callbacks.Callback):
    def __init__(self, images, labels, masks):
        super().__init__(); self.images, self.labels, self.masks, self.history = images, labels, masks, []
    def on_epoch_end(self, epoch, logs=None):
        probabilities = self.model.predict(self.images, verbose=0)
        scores = {}
        for index, condition in enumerate(SPECIALIST_CONDITIONS):
            known = self.masks[:, index].astype(bool)
            scores[condition] = float(f1_score(self.labels[known, index], probabilities[known, index] >= 0.50, zero_division=0))
        focus_f1 = float(np.mean(list(scores.values())))
        self.history.append({'focus_f1': focus_f1, **scores})
        if logs is not None: logs['val_specialist_f1'] = focus_f1
        print(f' validation specialist F1: {focus_f1:.4f}  pothole={scores["pothole"]:.4f}  manhole={scores["manhole_cover"]:.4f}')

specialist_callback = SpecialistValidationF1(validation_images, validation_labels, validation_masks)
best_path = OUTPUT_DIRECTORY / 'e8_pothole_manhole_specialist_best.keras'
fine_history = model.fit(train_dataset, validation_data=validation_dataset, epochs=15, verbose=1, callbacks=[
    specialist_callback,
    tf.keras.callbacks.ModelCheckpoint(best_path, monitor='val_specialist_f1', mode='max', save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor='val_specialist_f1', mode='max', patience=4, restore_best_weights=True),
])


In [ ]:
# Save validation evidence and package the Kaggle output. Download it from Kaggle after saving the notebook version.
import matplotlib.pyplot as plt

best_model = tf.keras.models.load_model(best_path, compile=False)
probabilities = best_model.predict(validation_images, verbose=1)
per_condition = {}
for index, condition in enumerate(SPECIALIST_CONDITIONS):
    known = validation_masks[:, index].astype(bool)
    precision, recall, f1, _ = precision_recall_fscore_support(validation_labels[known, index], probabilities[known, index] >= 0.50, average='binary', zero_division=0)
    per_condition[condition] = {'known_images': int(known.sum()), 'positive_images': int(validation_labels[known, index].sum()), 'precision': float(precision), 'recall': float(recall), 'f1': float(f1)}
metrics = {
    'experiment': 'v2_multilabel_e8_pothole_manhole_specialist', 'selection_data': 'validation only',
    'protected_test_loaded': False, 'model': 'ImageNet-pretrained EfficientNetV2B0 320x320 two-label specialist',
    'threshold': 0.50, 'specialist_macro_f1': float(np.mean([item['f1'] for item in per_condition.values()])),
    'head_epochs_completed': len(head_history.history['loss']), 'fine_tune_epochs_completed': len(fine_history.history['loss']), 'per_condition': per_condition
}
(OUTPUT_DIRECTORY / 'e8_validation_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
best_model.save(OUTPUT_DIRECTORY / 'e8_pothole_manhole_specialist_final.keras')
print(json.dumps(metrics, indent=2))

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(head_history.history['loss'] + fine_history.history['loss'], label='train loss')
axes[0].plot(head_history.history['val_loss'] + fine_history.history['val_loss'], label='validation loss')
axes[0].axvline(len(head_history.history['loss']) - 0.5, color='gray', linestyle='--', label='fine-tuning starts')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('masked focal loss'); axes[0].legend()
axes[1].plot([item['focus_f1'] for item in specialist_callback.history], label='pothole/manhole F1')
axes[1].set_xlabel('fine-tune epoch'); axes[1].set_ylabel('specialist F1'); axes[1].legend()
figure.tight_layout(); figure.savefig(OUTPUT_DIRECTORY / 'e8_learning_curve.png', dpi=150)

result_zip = WORK_ROOT / 'v2_multilabel_e8_pothole_manhole_specialist_output.zip'
if result_zip.exists(): result_zip.unlink()
shutil.make_archive(str(result_zip.with_suffix('')), 'zip', OUTPUT_DIRECTORY)
print('Completed output ZIP:', result_zip)
print('Protected test remains untouched.')
